In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import time
import numpy as np
from tqdm import tqdm
from torchmetrics.functional import calibration_error

In [2]:
# =========================================================
# DEVICE SETUP
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

✅ Using device: cuda


In [3]:
# =========================================================
# DATASET
# =========================================================
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2, pin_memory=True)

In [4]:
# =========================================================
# VGG16 Model for CIFAR-100
# =========================================================
class VGG16_CIFAR100(nn.Module):
    def __init__(self, num_classes=100):
        super(VGG16_CIFAR100, self).__init__()
        self.features = torchvision.models.vgg16_bn(weights=None).features
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(True),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1,1))
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model = VGG16_CIFAR100().to(device)
criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)


In [5]:
# =========================================================
# METRICS
# =========================================================
def topk_accuracy(output, target, topk=(1,)):
    maxk = max(topk)
    _, pred = output.topk(maxk, 1, True, True)
    correct = pred.eq(target.view(-1,1).expand_as(pred))
    res = []
    for k in topk:
        correct_k = correct[:, :k].sum().item()
        res.append(correct_k)
    return res

def mean_off_diagonal_covariance(features, eps=1e-8):
    X = features - features.mean(dim=0, keepdim=True)
    cov = (X.T @ X) / (X.shape[0]-1)
    var = cov.diag()
    denom = torch.sqrt(var.unsqueeze(0)*var.unsqueeze(1)).clamp_min(eps)
    corr = cov/denom
    corr.fill_diagonal_(0)
    return corr.abs().mean()

def linear_CKA(X, Y):
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    hsic = torch.norm(X.T @ Y, p='fro')**2
    var1 = torch.norm(X.T @ X, p='fro') * torch.norm(Y.T @ Y, p='fro')
    return (hsic / var1).item()


In [6]:
# =========================================================
# TRAINING
# =========================================================
num_epochs = 100
best_acc = 0
best_val_loss = float('inf')
patience = 5
no_improve_epochs = 0
train_losses, test_losses = [], []
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss, total, correct_top1, correct_top5 = 0.0, 0, 0, 0
    epoch_start = time.time()

    for inputs, labels in tqdm(trainloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        top1, top5 = topk_accuracy(outputs, labels, topk=(1, 5))
        correct_top1 += top1
        correct_top5 += top5
        total += labels.size(0)

    train_loss = running_loss / len(trainloader)
    train_acc1 = 100 * correct_top1 / total
    train_acc5 = 100 * correct_top5 / total
    train_losses.append(train_loss)

    # =====================================================
    # VALIDATION
    # =====================================================
    model.eval()
    correct_top1, correct_top5, total = 0, 0, 0
    test_loss = 0.0
    probs, targets, features = [], [], []

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            top1, top5 = topk_accuracy(outputs, labels, topk=(1, 5))
            correct_top1 += top1
            correct_top5 += top5
            total += labels.size(0)
            probs.append(F.softmax(outputs, dim=1).cpu())
            targets.append(labels.cpu())
            feats = model.features(inputs).view(inputs.size(0), -1).detach().cpu()
            features.append(feats)

    test_loss /= len(testloader)
    test_losses.append(test_loss)
    test_acc1 = 100 * correct_top1 / total
    test_acc5 = 100 * correct_top5 / total
    train_test_gap = abs(train_acc1 - test_acc1)

    all_probs = torch.cat(probs)
    all_targets = torch.cat(targets)
    ece = calibration_error(all_probs, all_targets, n_bins=15, task="multiclass", num_classes=100).item()
    all_features = torch.cat(features, dim=0)
    moc = mean_off_diagonal_covariance(all_features)

    epoch_time = time.time() - epoch_start
    print(f"\n📊 Epoch {epoch+1}:")
    print(f"Train Loss: {train_loss:.4f} | Train Acc@1: {train_acc1:.2f}% | Train Acc@5: {train_acc5:.2f}%")
    print(f"Val Loss: {test_loss:.4f} | Val Acc@1: {test_acc1:.2f}% | Val Acc@5: {test_acc5:.2f}%")
    print(f"Train–Test Gap: {train_test_gap:.2f}% | ECE: {ece:.4f} | MOC: {moc:.4f}")
    print(f"⏱ Epoch Time: {epoch_time:.2f}s")

    # Early stopping
    if test_loss < best_val_loss - 1e-4:
        best_val_loss = test_loss
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= patience:
            print(f"\n⏹ Early stopping triggered after {epoch+1} epochs.")
            break

    # Save best model
    if test_acc1 > best_acc:
        best_acc = test_acc1
        torch.save(model.state_dict(), "best_vgg16_cifar100_mps.pth")

total_time = time.time() - start_time
print(f"\n✅ Training Complete.")
print(f"Best Top-1 Accuracy: {best_acc:.2f}%")
print(f"Total Training Time: {total_time/60:.2f} minutes")

# =========================================================
# INFERENCE TIME
# =========================================================
model.eval()
dummy_input = torch.randn(1, 3, 32, 32).to(device)
start = time.time()
_ = model(dummy_input)
inference_time = time.time() - start
print(f"\n⚡ Inference Time (1 sample): {inference_time*1000:.3f} ms")

Epoch 1/100: 100%|██████████| 391/391 [00:10<00:00, 38.32it/s]



📊 Epoch 1:
Train Loss: 3.9560 | Train Acc@1: 8.04% | Train Acc@5: 27.65%
Val Loss: 3.8608 | Val Acc@1: 9.46% | Val Acc@5: 31.79%
Train–Test Gap: 1.42% | ECE: 0.0384 | MOC: 0.3046
⏱ Epoch Time: 11.67s


Epoch 2/100: 100%|██████████| 391/391 [00:09<00:00, 40.25it/s]



📊 Epoch 2:
Train Loss: 3.5202 | Train Acc@1: 14.19% | Train Acc@5: 41.08%
Val Loss: 3.5498 | Val Acc@1: 13.85% | Val Acc@5: 41.10%
Train–Test Gap: 0.34% | ECE: 0.0712 | MOC: 0.2954
⏱ Epoch Time: 11.41s


Epoch 3/100: 100%|██████████| 391/391 [00:09<00:00, 39.84it/s]


...


📊 Epoch 17:
Train Loss: 1.5007 | Train Acc@1: 57.47% | Train Acc@5: 86.41%
Val Loss: 1.8824 | Val Acc@1: 50.07% | Val Acc@5: 79.62%
Train–Test Gap: 7.40% | ECE: 0.0530 | MOC: 0.1725
⏱ Epoch Time: 10.94s


Epoch 18/100: 100%|██████████| 391/391 [00:10<00:00, 37.71it/s]



📊 Epoch 18:
Train Loss: 1.4578 | Train Acc@1: 58.63% | Train Acc@5: 86.84%
Val Loss: 1.7685 | Val Acc@1: 52.22% | Val Acc@5: 81.49%
Train–Test Gap: 6.41% | ECE: 0.0406 | MOC: 0.1693
⏱ Epoch Time: 11.96s


Epoch 19/100: 100%|██████████| 391/391 [00:11<00:00, 34.13it/s]



📊 Epoch 19:
Train Loss: 1.4099 | Train Acc@1: 59.80% | Train Acc@5: 87.44%
Val Loss: 1.9427 | Val Acc@1: 48.59% | Val Acc@5: 78.60%
Train–Test Gap: 11.21% | ECE: 0.0659 | MOC: 0.1741
⏱ Epoch Time: 13.21s

⏹ Early stopping triggered after 19 epochs.

✅ Training Complete.
Best Top-1 Accuracy: 54.01%
Total Training Time: 3.72 minutes

⚡ Inference Time (1 sample): 17.586 ms
